# Import packages

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, sys
from pathlib import Path
for p in [Path.cwd()] + list(Path.cwd().parents):
    if p.name == 'Multifirefly-Project':
        os.chdir(p)
        sys.path.insert(0, str(p / 'multiff_analysis/multiff_code/methods'))
        break
    
from data_wrangling import specific_utils, process_monkey_information, general_utils, combine_info_utils
from pattern_discovery import pattern_by_trials, pattern_by_trials, cluster_analysis, organize_patterns_and_features
from visualization.matplotlib_tools import plot_behaviors_utils
from neural_data_analysis.neural_analysis_tools.get_neural_data import neural_data_processing
from neural_data_analysis.neural_analysis_tools.visualize_neural_data import plot_neural_data, plot_modeling_result
from neural_data_analysis.neural_analysis_tools.model_neural_data import transform_vars, neural_data_modeling, drop_high_corr_vars, drop_high_vif_vars
from neural_data_analysis.topic_based_neural_analysis.neural_vs_behavioral import prep_monkey_data, prep_target_data, neural_vs_behavioral_class
from neural_data_analysis.topic_based_neural_analysis.planning_and_neural import planning_and_neural_class, pn_utils, pn_helper_class, pn_aligned_by_seg, pn_aligned_by_event
from neural_data_analysis.neural_analysis_tools.cca_methods import cca_class
from neural_data_analysis.neural_analysis_tools.cca_methods import cca_class, cca_utils, cca_cv_utils
from neural_data_analysis.neural_analysis_tools.cca_methods.cca_plotting import cca_plotting, cca_plot_lag_vs_no_lag, cca_plot_cv
from machine_learning.ml_methods import regression_utils, regz_regression_utils, ml_methods_class, classification_utils, ml_plotting_utils, ml_methods_utils
from planning_analysis.show_planning import nxt_ff_utils, show_planning_utils
from neural_data_analysis.neural_analysis_tools.gpfa_methods import elephant_utils, fit_gpfa_utils, plot_gpfa_utils, gpfa_helper_class
from neural_data_analysis.neural_analysis_tools.align_trials import time_resolved_regression, time_resolved_gpfa_regression,plot_time_resolved_regression
from neural_data_analysis.neural_analysis_tools.align_trials import align_trial_utils
from decision_making_analysis.event_detection import detect_rsw_and_rcap

from neural_data_analysis.topic_based_neural_analysis.stop_event_analysis.stop_psth import core_stops_psth, psth_postprocessing, psth_stats, compare_events, dpca_utils
from neural_data_analysis.topic_based_neural_analysis.stop_event_analysis.get_stop_events import get_stops_utils, collect_stop_data

from neural_data_analysis.neural_analysis_tools.glm_tools.glm_fit import general_glm_fit, cv_stop_glm, glm_fit_utils, variance_explained, glm_runner
from neural_data_analysis.topic_based_neural_analysis.stop_event_analysis.stop_glm.glm_plotting import plot_spikes, plot_glm_fit, plot_tuning_func, compare_glm_fit
from neural_data_analysis.design_kits.design_around_event import event_binning, stop_design, cluster_design, design_checks
from neural_data_analysis.topic_based_neural_analysis.stop_event_analysis.stop_glm.glm_hyperparams import compare_glm_configs, glm_hyperparams_class
from neural_data_analysis.topic_based_neural_analysis.ff_visibility import ff_vis_epochs, vis_design

from neural_data_analysis.topic_based_neural_analysis.stop_event_analysis.get_stop_events import decode_stops_design

# import decoding
from neural_data_analysis.neural_analysis_tools.decoding_tools.event_decoding import decoding_utils, decoding_analysis, plot_decoding, cmp_decode, load_results
from neural_data_analysis.design_kits.design_by_segment import spike_history
from neural_data_analysis.neural_analysis_tools.decoding_tools.general_decoding import cv_decoding
from neural_data_analysis.neural_analysis_tools.encoding_tools.encoding_by_topics import encode_stops_pipeline, encode_vis_pipeline, encode_pn_pipeline
from neural_data_analysis.topic_based_neural_analysis.replicate_one_ff.one_ff_gam import one_ff_gam_fit, gam_variance_explained
from neural_data_analysis.topic_based_neural_analysis.stop_event_analysis import stop_parameters

from neural_data_analysis.topic_based_neural_analysis.replicate_one_ff.one_ff_gam import one_ff_gam_fit, one_ff_gam_design, penalty_tuning, backward_elimination, gam_variance_explained, plot_gam_fit
from data_wrangling import combine_info_utils

import sys
import math
import gc
import subprocess
from pathlib import Path

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rc
from scipy import linalg, interpolate
from scipy.signal import fftconvolve
from scipy.io import loadmat
from scipy import sparse
import torch
from numpy import pi
import cProfile
import pstats
import json

# Machine Learning imports
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.multivariate.cancorr import CanCorr
import statsmodels.api as sm

# Neuroscience specific imports
import neo
import rcca

# To fit gpfa
import numpy as np
from importlib import reload
from scipy.integrate import odeint
import quantities as pq
import neo
from elephant.spike_train_generation import inhomogeneous_poisson_process
from elephant.gpfa import GPFA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from elephant.gpfa import gpfa_core, gpfa_util

plt.rcParams["animation.html"] = "html5"
os.environ['KMP_DUPLICATE_LIB_OK']='True'
rc('animation', html='jshtml')
matplotlib.rcParams.update(matplotlib.rcParamsDefault)
matplotlib.rcParams['animation.embed_limit'] = 2**128
pd.set_option('display.float_format', lambda x: '%.5f' % x)
np.set_printoptions(suppress=True)
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)

print("done")


%load_ext autoreload
%autoreload 2
%matplotlib inline

pd.set_option('display.max_colwidth', 200)

# Encode stops

## init the class

In [ ]:
# raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0316"
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0222"

data_exists_ok = False

prs = stop_parameters.default_prs()

runner = encode_stops_pipeline.StopEncodingRunner(raw_data_folder_path=raw_data_folder_path, 
                                                  bin_width=0.04, 
                                                  t_max=0.20,
                                                  stop_prs = prs)
runner._collect_data(exists_ok=data_exists_ok,
                     tuning_feature_mode='boxcar_only')

## category_variance

In [ ]:
unit_idx = 0

In [ ]:
contrib = runner.run_category_variance_contributions(
    unit_idx,
    n_folds=5,
    buffer_samples=20,
    load_if_exists=True,  # load cached if available
)
# contrib['full_cv_result'], contrib['category_contributions'], contrib['category_contributions_csv']

In [ ]:
df_plot = plot_gam_fit.plot_category_variance_contributions(contrib, sort_by="delta_pseudo_r2")
df_plot

### collect all neurons' info

In [ ]:

# unit_indices = range(128)  # or whatever you have
unit_indices = range(30)  # or whatever you have
all_results = []
for u in unit_indices:
    try:
        result = runner.run_category_variance_contributions(
            unit_idx=u,
            retrieve_only=True,   # use cached if already computed
        )
        all_results.append(result)
    except Exception as e:
        #print(f"Error for unit {u}: {e}. Will stop trying further units.")
        break
    
all_results = plot_gam_fit.add_clipped_leave_delta_metric(all_results)
plot_gam_fit.plot_category_ecdf(
    all_results,
    metric_key='delta_pseudo_r2_clip_leave',
    #category_groups=category_groups,
    log_x=False,
)

## penalty tuning

Here, the cv_score in run_penalty_tuning is the mean held-out Poisson log-likelihood from cross-validation. The best lambdas are those with the highest mean held-out log-likelihood.

In [ ]:
unit_idx = 0

In [ ]:
tuning = runner.run_penalty_tuning(
    unit_idx,
    n_folds=5,
    lam_grid=None,  # default grid; or e.g. {'lam_f': [10,100], 'lam_g': [1,10], ...}
    group_name_map=None,  # default groups; or map lam keys -> group names
    load_if_exists=True,
)
# tuning['best_lams'], tuning['cv_results'], tuning['save_path']

In [ ]:
df, df_sorted = penalty_tuning.analyze_penalty_tuning(tuning)

## backward elimination

In [ ]:
elim = runner.run_backward_elimination(
    unit_idx,
    alpha=0.05,
    n_folds=10,
    load_if_exists=True,
)
# elim['kept_groups'], elim['history'], elim['history_csv'], elim['save_path']

In [ ]:
elim_res = runner.run_backward_elimination(unit_idx=12)

print([g.name for g in elim_res["kept_groups"]])
print(elim_res["history_csv"])

In [ ]:
eliminated = [h["removed"] for h in elim_res["history"]]
print(eliminated)

In [ ]:
stop!

## run gam

In [ ]:
unit_idx = 0
groups, lambda_config = runner.get_stop_gam_groups(unit_idx=unit_idx, lam_f=100.0, lam_g=10.0, lam_h=10.0, lam_p=10.0)

fit_result = runner.fit_stop_poisson_gam(
    unit_idx=unit_idx,
)

plot_gam_fit.plot_variables(fit_result.coef, runner.structured_meta_groups)  

## cv tuning curves

In [ ]:
for unit_idx in range(runner.num_neurons):
    groups, lambda_config = runner.get_stop_gam_groups(unit_idx=unit_idx, lam_f=100.0, lam_g=10.0, lam_h=10.0, lam_p=10.0)
    fold_coef_results = runner.crossval_stop_tuning_curve_coef(unit_idx, groups)
    plot_gam_fit.plot_fold_coefficient_by_variable(fold_coef_results, runner.structured_meta_groups, var_list=None, use_sem=True)

# Crossval

## iterate through neurons

In [ ]:
data_exists_ok = False

prs = stop_parameters.default_prs()

runner = encode_stops_pipeline.StopEncodingRunner(raw_data_folder_path=raw_data_folder_path, 
                                                  bin_width=0.04, 
                                                  t_max=0.20,
                                                  stop_prs = prs)
runner._collect_data(exists_ok=data_exists_ok,
                     tuning_feature_mode='boxcar_only')

In [ ]:
all_neuron_r2 = runner.crossval_stop_variance_explained_all_neurons(
    lam_f=100.0, lam_g=10.0, lam_h=10.0, lam_p=10.0,
    n_folds=5,
    load_if_exists=False,
    cv_mode='blocked_time_buffered',
    buffer_samples=20,
    verbose=True,
    plot_cdf=True,
    log_x=False,
)

In [ ]:
## use loop instead

# load_if_exists = True

# all_neuron_r2 = []
# lam_suffix = one_ff_gam_fit.generate_lambda_suffix(lambda_config=lambda_config)

# for unit_idx in np.arange(runner.binned_spikes.shape[1]):
    
#     outdir = runner.base / f'neuron_{unit_idx}'
#     save_path = outdir / 'cv_var_explained' / f'{lam_suffix}.pkl'
#     cv_save_path = str(outdir / 'cv_var_explained' / f'{lam_suffix}.pkl')

#     # Cross-validated variance explained
#     cv_res = runner.crossval_stop_variance_explained(
#         unit_idx=unit_idx,
#         groups=groups,
#         n_folds=5,
#         fit_kwargs=dict(l1_groups=[], max_iter=1000, tol=1e-6, verbose=False, save_path=None),
#         save_path=cv_save_path,
#         load_if_exists=load_if_exists,
#         cv_mode='blocked_time_buffered',
#         buffer_samples=20,
#     )
#     print(cv_res['mean_classical_r2'], cv_res['mean_pseudo_r2'])
#     all_neuron_r2.append(cv_res["mean_pseudo_r2"])
    
# gam_variance_explained.plot_variance_explained_cdf(all_neuron_r2, log_x=False)

In [ ]:
all_neuron_r2

## Iterate through all sessions

In [ ]:
reload(encode_stops_pipeline)

In [ ]:
load_only = True

raw_data_dir_name = 'all_monkey_data/raw_monkey_data'
monkey_names = ['monkey_Bruno', 'monkey_Schro']  # or ['monkey_Bruno'] for one monkey
tuning_feature_mode = 'boxcar_only'
load_if_exists = True

all_session_results = {}  # raw_data_folder_path -> all_neuron_r2

for monkey_name in monkey_names:
    sessions_df = combine_info_utils.make_sessions_df_for_one_monkey(
        raw_data_dir_name, monkey_name
    )
    for _, row in sessions_df.iterrows():
        raw_data_folder_path = os.path.join(
            raw_data_dir_name, row['monkey_name'], row['data_name']
        )
        print('=' * 80)
        print(f'Processing: {raw_data_folder_path}')
        print('=' * 80)
        try:
            prs = stop_parameters.default_prs()
            runner = encode_stops_pipeline.StopEncodingRunner(
                raw_data_folder_path=raw_data_folder_path,
                bin_width=0.04,
                t_max=0.20,
                stop_prs=prs,
            )
            all_neuron_r2 = runner.crossval_stop_variance_explained_all_neurons(
                lam_f=100.0, lam_g=10.0, lam_h=10.0, lam_p=10.0,
                n_folds=5,
                load_if_exists=load_if_exists,
                cv_mode='blocked_time_buffered',
                buffer_samples=20,
                verbose=True,
                plot_cdf=False,  # set True to plot per session
                log_x=False,
                load_only=load_only,
            )

            print('all_neuron_r2', all_neuron_r2)
            if all_neuron_r2 is not None:
                all_session_results[raw_data_folder_path] = all_neuron_r2
                
                gam_variance_explained.plot_variance_explained_cdf(all_neuron_r2, log_x=False)
        except Exception as e:
            print(f'[ERROR] Failed for {raw_data_folder_path}: {e}')
            continue

## on one neuron

In [ ]:
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0219"

data_exists_ok = False

prs = stop_parameters.default_prs()

runner = encode_stops_pipeline.StopEncodingRunner(raw_data_folder_path=raw_data_folder_path, 
                                                  bin_width=0.04, 
                                                  t_max=0.20,
                                                  stop_prs = prs)
runner._collect_data(exists_ok=data_exists_ok,
                     tuning_feature_mode='boxcar_only')

In [ ]:
# do crossval on one neuron
unit_idx = 17
# After runner._collect_data(exists_ok=True)
runner.get_stop_gam_save_paths(unit_idx=unit_idx)

# Cross-validated variance explained
cv_res = runner.crossval_stop_variance_explained(
    unit_idx=unit_idx,
    groups=groups,
    n_folds=5,
    fit_kwargs=dict(l1_groups=[], max_iter=1000, tol=1e-6, verbose=False, save_path=None),
    save_path=runner.fit_save_path,
    load_if_exists=False,
    cv_mode='blocked_time_buffered',
    buffer_samples=20,
)
print(cv_res['mean_classical_r2'], cv_res['mean_pseudo_r2'])

# debug

In [ ]:
groups, config = runner.get_gam_groups(0)
groups

In [ ]:
groups, config = runner.get_gam_groups(0)
groups

# VIS

In [ ]:
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0220"

In [ ]:
# Vis encoding
runner = encode_vis_pipeline.FFVisEncodingRunner(raw_data_folder_path, bin_width=0.04, t_max=0.20)
runner._collect_data(exists_ok=True)
fit = runner.fit_poisson_gam(unit_idx=0)
r2_list = runner.crossval_variance_explained_all_neurons(n_folds=5)

In [ ]:

# unit_indices = range(128)  # or whatever you have
unit_indices = range(30)  # or whatever you have
all_results = []
for u in unit_indices:
    try:
        result = runner.run_category_variance_contributions(
            unit_idx=u,
            retrieve_only=True,   # use cached if already computed
        )
        all_results.append(result)
    except Exception as e:
        #print(f"Error for unit {u}: {e}. Will stop trying further units.")
        break
    
all_results = plot_gam_fit.add_clipped_leave_delta_metric(all_results)
plot_gam_fit.plot_category_ecdf(
    all_results,
    metric_key='delta_pseudo_r2_clip_leave',
    #category_groups=category_groups,
    log_x=False,
)

## pen tune

In [ ]:
tuning = runner.run_penalty_tuning(
    unit_idx,
    n_folds=5,
    lam_grid=None,  # default grid; or e.g. {'lam_f': [10,100], 'lam_g': [1,10], ...}
    group_name_map=None,  # default groups; or map lam keys -> group names
    load_if_exists=True,
)
# tuning['best_lams'], tuning['cv_results'], tuning['save_path']

# PN

In [ ]:
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0222"

In [ ]:
# PN encoding
runner = encode_pn_pipeline.PNEncodingRunner(raw_data_folder_path, bin_width=0.04, t_max=0.20)
runner._collect_data(exists_ok=True)
fit = runner.fit_poisson_gam(unit_idx=0, lam_f=100, lam_g=10, lam_h=10, lam_p=10)
r2_list = runner.crossval_variance_explained_all_neurons(n_folds=5)


In [ ]:
# unit_indices = range(128)  # or whatever you have
unit_indices = range(30)  # or whatever you have
all_results = []
for u in unit_indices:
    try:
        result = runner.run_category_variance_contributions(
            unit_idx=u,
            retrieve_only=True,   # use cached if already computed
        )
        all_results.append(result)
    except Exception as e:
        #print(f"Error for unit {u}: {e}. Will stop trying further units.")
        break
    
all_results = plot_gam_fit.add_clipped_leave_delta_metric(all_results)
plot_gam_fit.plot_category_ecdf(
    all_results,
    metric_key='delta_pseudo_r2_clip_leave',
    #category_groups=category_groups,
    log_x=False,
)

## check design_df

In [ ]:
design_df = runner.get_design_for_unit(1)
design_df.columns


# Appendix

In [ ]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

In [ ]:
runner.binned_feats.shape

In [ ]:
runner.binned_feats.describe().T

In [ ]:
runner.binned_feats.describe().T.sort_values(by='std', ascending=False)

## check for NA

In [ ]:
na_rows, na_cols = general_utils.check_na_in_df(runner.design_df_w_history, df_name="DataFrame", return_rows_and_columns=True)

In [ ]:
runner.design_df_w_history.describe().T

In [ ]:
na_rows

In [ ]:
df = runner.design_df_w_history[['event_t_from_cluster_start_s']].copy()
df

In [ ]:
runner.design_df_w_history['event_t_from_cluster_start_s']

In [ ]:
runner.design_df_w_history.columns.tolist()

## check cluster info

In [ ]:
df = runner.stop_meta_used
df

In [ ]:
# Meta columns that contain cluster info
meta_cluster_cols = [c for c in runner.stop_meta_used.columns if 'cluster' in c.lower() or c == 'is_clustered']
runner.stop_meta_used[['event_id', 'k_within_seg'] + meta_cluster_cols]

In [ ]:
# Counts
runner.binned_feats['is_clustered'].value_counts()
runner.binned_feats['event_cluster_id'].nunique()

# Rows with no cluster (event_cluster_id NaN)
runner.binned_feats['event_cluster_id'].isna().sum()

# Cluster size distribution (clustered events only)
runner.binned_feats.loc[runner.binned_feats['is_clustered'] == 1, 'event_cluster_size'].value_counts()

In [ ]:
df = runner.get_design_for_unit(0)  # NaN columns already filled with 0
cluster_cols = [c for c in df.columns if 'cluster' in c.lower() or c == 'is_clustered']
df[cluster_cols].describe()

## debug stop_category_df

In [ ]:
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0221"
bin_width = 0.04


In [ ]:
runner.pn.stop_category_df

In [ ]:
pn, datasets, new_seg_info, events_with_stats = \
    decode_stops_design._prepare_stop_design_inputs(
        raw_data_folder_path, bin_width)

In [ ]:
pn.stop_category_df

In [ ]:
unique_stops_df = get_stops_utils.extract_unique_stops(pn.monkey_information)
unique_stops_df